# 第11回 多変量時系列

このノートブックは、スライドと同じ順に並んでいます。
コードはすべて完成しているので、上から順に実行して、結果を読んでください。

- 0節：セットアップ（授業の最初に実行する）
- A節：理論をシミュレーションで確かめる
- B節：実データを解析する
- C節：設定を変えて回し直す

今日の問い：ほかの系列の過去は、自分の予測に役立つか。

## 0. セットアップ

授業が始まったら、まずこのセルを実行してください。
パッケージのインストールに少し時間がかかります。終わるまで、次のセルは動きません。

In [ ]:
# OSのコードネームを確認（noble = Ubuntu 24.04 など）
system("lsb_release -cs", intern = TRUE)

options(
  repos = c(CRAN = "https://packagemanager.posit.co/cran/__linux__/noble/2026-09-01"),
  HTTPUserAgent = sprintf("R/%s R (%s)", getRversion(),
    paste(getRversion(), R.version["platform"], R.version["arch"], R.version["os"]))
)
install.packages(c("tsibble", "fable", "feasts", "fpp3", "vars"))
sessionInfo()

## A. 理論をシミュレーションで確かめる

In [ ]:
library(tsibble)
library(feasts)
library(fable)
library(ggtime)   # 相互相関とACFの図（autoplot）
library(dplyr)
library(ggplot2)
options(repr.plot.width = 8, repr.plot.height = 4.5)
invisible(Sys.setlocale("LC_TIME", "C"))   # 図の月の名前を英語にする

`vars`パッケージは、`library()`で読み込まず、`vars::VAR()`のように呼びます。
読み込むと、`dplyr`の`select()`が別のパッケージの同じ名前の関数に隠れてしまうからです。

### A-1. 2本の系列を重ねる

これまでの回は、1本の系列を扱いました。今日は、2本の系列 $x_t$ と $y_t$ を同時に扱います。

人工の系列を2本作ります。作り方は次のとおりです。

$$
\begin{aligned}
x_t &= 0.6\,x_{t-1} + w_{x,t} \\
y_t &= 0.4\,x_{t-1} + 0.5\,y_{t-1} + w_{y,t}
\end{aligned}
$$

- $y_t$ の式には、$x$ の過去（$x_{t-1}$）が入っています。
- $x_t$ の式には、$y$ の過去が入っていません。
- 影響は、$x$ から $y$ への片方向だけです。
- ノイズ $w_{x,t}$ と $w_{y,t}$ は、同じ時点どうしで相関0.5をもたせます（違う時点どうしは無相関）。

係数を行列にまとめます。行が「どの系列の式か」、列が「どの系列の過去か」です。

In [ ]:
# VAR(1)の系列を作る。w はノイズの行列（1行が1時点）、Phi は係数の行列。最初の100時点は捨てる
sim_var1 <- function(w, Phi) {
  x <- matrix(0, nrow(w), ncol(w))
  for (t in 2:nrow(w)) x[t, ] <- Phi %*% x[t - 1, ] + w[t, ]
  x[-(1:100), , drop = FALSE]
}
# VAR(1)の係数を、最小二乗法で推定する。Z は系列の行列（1列が1本）。返す行列の1列目は定数項
fit_var1 <- function(Z) {
  X <- cbind(const = 1, Z[-nrow(Z), , drop = FALSE])           # 1時点前の値
  t(solve(crossprod(X), crossprod(X, Z[-1, , drop = FALSE])))  # 式ごとの最小二乗推定量
}
# 推定した係数 B で、最後の値 last から h 時点先まで予測する
forecast_var1 <- function(B, last, h) {
  out <- matrix(NA, h, length(last))
  for (i in 1:h) {
    last <- B[, 1] + B[, -1, drop = FALSE] %*% last
    out[i, ] <- last
  }
  out
}
rmse <- function(estimate, truth) sqrt(mean((estimate - truth)^2))

In [ ]:
set.seed(1)
n <- 300
Phi <- matrix(c(0.6, 0.0,
                0.4, 0.5), 2, 2, byrow = TRUE,
              dimnames = list(c("x", "y"), c("x", "y")))     # x から y へだけ効く
Sigma <- matrix(c(1.0, 0.5,
                  0.5, 1.0), 2, 2, dimnames = dimnames(Phi)) # ノイズの分散と共分散
w <- matrix(rnorm(2 * (n + 100)), ncol = 2) %*% chol(Sigma)  # 同じ時点どうしが相関するノイズ
Z <- sim_var1(w, Phi)
colnames(Z) <- c("x", "y")
sim <- tsibble(t = 1:n, x = Z[, "x"], y = Z[, "y"], index = t)
sim_long <- data.frame(t = rep(1:n, 2), series = rep(c("x", "y"), each = n), value = c(sim$x, sim$y))
ggplot(sim_long |> filter(t <= 100), aes(t, value, colour = series)) +
  geom_line() +
  labs(x = "Time t (step)", y = "Value", colour = "Series")

2本は、似た動きをしています。ただし、図を見ただけでは、どちらが先に動いているのかは分かりません。

### A-2. 相互相関

2本の系列の、時点をずらした相関を、相互相関関数（cross-correlation function, CCF）といいます。

$$
\rho_{xy}(h) = \mathrm{Corr}(x_t,\ y_{t+h})
$$

- $h > 0$：$x$ と、それより $h$ 時点あとの $y$ の相関（$x$ が先）
- $h < 0$：$x$ と、それより前の $y$ の相関（$y$ が先）

In [ ]:
sim |>
  CCF(x, y, lag_max = 10) |>
  autoplot() +
  labs(x = "Lag h (step)", y = "Sample CCF: Corr(x_t, y_{t+h})")
# 確かめ：ラグ +1 と -1 の相関を、直接計算する
c(lag_plus_1  = cor(sim$x[1:(n - 1)], sim$y[2:n]),    # x が先
  lag_minus_1 = cor(sim$x[2:n], sim$y[1:(n - 1)]))    # y が先

- 山は右（正のラグ）に寄っています。$x$ が先に動き、$y$ があとを追う形です。
- ただし、負のラグの相関も0ではありません。$y$ から $x$ へは効かないように作ったのに、です。
- $x$ も $y$ も、それぞれ自己相関をもちます。相互相関には、2本の関係と、それぞれの自己相関が混ざっています。

第1回の見せかけの回帰を思い出します。互いに無関係な2本でも、相関が「有意」に見えることがありました。
無関係な2本を作り、ラグ0の相関が青い破線（$\pm 1.96/\sqrt{n}$）の外に出る割合を出します。系列は3種類です。

- `white_noise`：ホワイトノイズ（iid）
- `ar1`：AR(1)（$\phi = 0.8$）。定常だが、自己相関がある
- `random_walk`：ランダムウォーク。定常でない

In [ ]:
set.seed(1)
make <- list(white_noise = function(n) rnorm(n),
             ar1         = function(n) as.numeric(stats::filter(rnorm(n + 100), 0.8, method = "recursive"))[-(1:100)],
             random_walk = function(n) cumsum(rnorm(n)))
rate_out <- sapply(make, function(f)
  mean(replicate(1000, abs(cor(f(200), f(200))) > 1.96 / sqrt(200))))
round(rate_out, 3)

- ホワイトノイズ：ほぼ5%です。破線は、iidを前提にした線です。
- AR(1)：定常でも、自己相関があるだけで、割合は5%よりずっと大きくなります。
- ランダムウォーク：ほとんどの場合に、外に出ます。第1回の見せかけの回帰と同じことが起きています。

2本の関係を調べるときは、まず定常な系列に直します。そのうえで、それぞれの自己相関を取り除いて、相手の過去が効いているかを見ます。
そのためのモデルが、次のVARです。

### A-3. VARモデル

2本の系列を、互いの過去で説明するモデルを、ベクトル自己回帰（vector autoregressive, VAR）モデルといいます。
1時点前までを使うものが、VAR(1)です。

$$
\begin{aligned}
x_t &= c_x + \phi_{xx}\,x_{t-1} + \phi_{xy}\,y_{t-1} + w_{x,t} \\
y_t &= c_y + \phi_{yx}\,x_{t-1} + \phi_{yy}\,y_{t-1} + w_{y,t}
\end{aligned}
$$

- どちらの式も、右辺は同じ（2本の過去）です。2本を対等に扱います。
- 第5回の回帰は、説明する側と、説明される側を、先に決めました。VARは決めません。

係数は、式ごとに最小二乗法で推定できます。自分で計算した値（`fit_var1`）と、`vars`パッケージの値を比べます。

In [ ]:
B_hat <- fit_var1(Z)                                    # 自分で計算（1列目は定数項）
round(B_hat, 3)
fit_sim <- vars::VAR(as.data.frame(Z), p = 1, type = "const")
round(vars::Bcoef(fit_sim), 3)                          # vars パッケージ（最後の列が定数項）
Phi                                                     # 作ったときの係数

- 2つの計算は、同じ値になります。
- 推定した係数は、作ったときの値に近くなります。$x$ の式の $y$ の過去の係数（本当は0）は、0の近くです。

VARモデルが定常であるための条件は、係数の行列 $\boldsymbol{\Phi}$ の固有値の絶対値が、すべて1より小さいことです。
第3回の、AR(1)の条件 $|\phi| < 1$ にあたります。

In [ ]:
Mod(eigen(Phi)$values)      # 作ったときの係数
vars::roots(fit_sim)        # 推定した係数

何時点前までを使うか（次数 $p$）は、第3回と同じく、AICやBICが最も小さいものを選びます。
`vars::VARselect()`の表では、`SC(n)`がBICです。

In [ ]:
sel_sim <- vars::VARselect(as.data.frame(Z), lag.max = 6, type = "const")
round(sel_sim$criteria[c("AIC(n)", "SC(n)"), ], 3)
sel_sim$selection[c("AIC(n)", "SC(n)")]

### A-4. Granger因果

$y$ の予測に、$y$ 自身の過去に加えて、$x$ の過去が役に立つとき、「$x$ から $y$ へGranger因果（Granger causality）がある」といいます。

- 名前に「因果」とありますが、原因と結果の関係を示すものではありません。「相手の過去が、自分の予測に役立つか」だけを見ています。
- VAR(1)では、「$x$ から $y$ へのGranger因果がない」は、$\phi_{yx} = 0$ と同じです。

検定では、2つの回帰を比べます。

- 小さい式：$y_t$ を、$y$ の過去だけで説明する
- 大きい式：$y_t$ を、$y$ の過去と $x$ の過去で説明する

$x$ の過去を入れて、残差の2乗和がはっきり減れば、帰無仮説「Granger因果がない」を棄却します（$F$ 検定）。

In [ ]:
# Granger因果の検定（帰無仮説：cause の過去は、もう片方の予測に役立たない）
granger <- function(fit, cause) {
  g <- vars::causality(fit, cause = cause)$Granger
  data.frame(cause = cause, F = unname(g$statistic[1, 1]), df1 = unname(g$parameter[1]),
             df2 = unname(g$parameter[2]), p_value = unname(g$p.value[1, 1]))
}

In [ ]:
rbind(granger(fit_sim, "x"),      # x から y へ（本当は、ある）
      granger(fit_sim, "y"))      # y から x へ（本当は、ない）
# 確かめ：2つの回帰を比べる F 検定を、自分で行う（x から y へ）
lagged <- data.frame(y = Z[-1, "y"], y_lag = Z[-n, "y"], x_lag = Z[-n, "x"])
anova(lm(y ~ y_lag, lagged), lm(y ~ y_lag + x_lag, lagged))

- $x$ から $y$ へ：p値はとても小さく、棄却されます。
- $y$ から $x$ へ：棄却されません。作ったとおりです。
- 自分で行った $F$ 検定の統計量は、`vars`の値と同じです（p値は、自由度の数え方の違いで、わずかに違います）。

1本だけでは、たまたまかもしれません。500回くり返して、5%水準で棄却された割合を出します。

In [ ]:
set.seed(1)
p_rep <- replicate(500, {
  Zr <- sim_var1(matrix(rnorm(2 * (n + 100)), ncol = 2) %*% chol(Sigma), Phi)
  colnames(Zr) <- c("x", "y")
  f <- vars::VAR(as.data.frame(Zr), p = 1, type = "const")
  c(x_to_y = granger(f, "x")$p_value, y_to_x = granger(f, "y")$p_value)
})
rowMeans(p_rep < 0.05)

- $x$ から $y$ へ：ほぼ毎回、棄却されます。
- $y$ から $x$ へ：棄却されるのは、約5%です。Granger因果がないときに、検定が約束どおりに働いています。

次に、Granger因果が「因果」ではないことを確かめます。3本目の系列 $z_t$ が、$x$ と $y$ の両方を動かしている場合です。

$$
x_t = z_{t-1} + w_{x,t}, \qquad y_t = z_{t-2} + w_{y,t}
$$

$x$ と $y$ のあいだに、直接の関係はありません。$z$ の動きが、$x$ には1時点あと、$y$ には2時点あとに現れるだけです。

In [ ]:
set.seed(1)
p_third <- replicate(500, {
  z <- as.numeric(stats::filter(rnorm(n + 2), 0.5, method = "recursive"))   # 両方を動かす系列
  d <- data.frame(x = z[2:(n + 1)] + rnorm(n), y = z[1:n] + rnorm(n))       # x は z の1時点あと、y は2時点あと
  f <- vars::VAR(d, p = 1, type = "const")
  # z を知っているとき：y の式に z の過去を入れたうえで、x の過去が役に立つか
  dz <- data.frame(y = d$y[-1], y_lag = d$y[-n], x_lag = d$x[-n], z_lag1 = z[3:(n + 1)], z_lag2 = z[2:n])
  c(x_to_y = granger(f, "x")$p_value,
    x_to_y_given_z = anova(lm(y ~ y_lag + z_lag1 + z_lag2, dz),
                           lm(y ~ y_lag + z_lag1 + z_lag2 + x_lag, dz))$`Pr(>F)`[2])
})
rowMeans(p_third < 0.05)

- `x_to_y`：$x$ から $y$ へのGranger因果が、ほぼ毎回「ある」と出ます。$x$ を動かしても $y$ は変わらないのに、です。
- `x_to_y_given_z`：$z$ の過去を式に入れると、$x$ の過去はもう役に立ちません（約5%）。

$x$ の過去は、$z$ の過去の手がかりになっているだけです。Granger因果は、予測に役立つかどうかを言うもので、原因かどうかは言いません。

### A-5. インパルス応答

第3回で、1回のショックが $j$ 時点あとにどれだけ残るかを $\psi_j$ と書きました。AR(1)では $\psi_j = \phi^j$ でした。
VAR(1)では、これが行列になります。

$$
\boldsymbol{\Psi}_j = \boldsymbol{\Phi}^j
$$

$\boldsymbol{\Psi}_j$ の $(i, k)$ 成分は、「系列 $k$ へのショックが、$j$ 時点あとの系列 $i$ に残る大きさ」です。
これを $j$ の関数としてみたものを、インパルス応答関数（impulse response function, IRF）といいます。

In [ ]:
# 理論のインパルス応答。Phi は係数の行列、P はショックの入り方（単位行列なら、1本だけに大きさ1のショック）
irf_theory <- function(Phi, P = diag(nrow(Phi)), n_ahead = 10) {
  nm <- rownames(Phi)
  out <- list()
  Psi <- diag(nrow(Phi))
  for (j in 0:n_ahead) {
    R <- Psi %*% P        # (i, k) 成分：k へのショックが、j 時点あとの i に残る大きさ
    out[[j + 1]] <- data.frame(j = j, impulse = rep(nm, each = length(nm)),
                               response = rep(nm, times = length(nm)), value = as.numeric(R))
    Psi <- Phi %*% Psi
  }
  do.call(rbind, out)
}
# 直交化したショックの入り方。order は変数の並べ順（先に置いた変数のショックは、同じ時点で、後の変数も動かす）
ortho_shock <- function(Sigma, order = 1:nrow(Sigma)) {
  P <- t(chol(Sigma[order, order]))     # 下三角行列。P %*% t(P) が Sigma に戻る
  back <- order(order)
  P[back, back]                         # 行と列を、もとの並べ順に戻す
}
# vars::irf() の結果を、図にしやすい表にする
irf_long <- function(ir) {
  do.call(rbind, lapply(ir$impulse, function(k) do.call(rbind, lapply(ir$response, function(i)
    data.frame(j = seq_len(nrow(ir$irf[[k]])) - 1, impulse = k, response = i,
               value = ir$irf[[k]][, i], lower = ir$Lower[[k]][, i], upper = ir$Upper[[k]][, i])))))
}
# 表に、パネルの名前（どの系列へのショックが、どの系列に伝わるか）の列を足す。列がショックを受ける系列、行が伝わる先
irf_panel <- function(d) {
  lv <- unique(d$impulse)
  d$panel <- factor(sprintf("shock to %s -> %s", d$impulse, d$response),
                    levels = as.vector(outer(lv, lv, function(a, b) sprintf("shock to %s -> %s", a, b))))
  d
}
# インパルス応答の図。表に order の列があれば色で分け、lower と upper の列があれば帯を描く
plot_irf <- function(d) {
  d <- irf_panel(d)
  g <- if ("order" %in% names(d)) ggplot(d, aes(j, value, colour = order)) + labs(colour = "Ordering") else ggplot(d, aes(j, value))
  if ("lower" %in% names(d)) g <- g + geom_ribbon(aes(ymin = lower, ymax = upper), alpha = 0.2, colour = NA)
  g + geom_hline(yintercept = 0, colour = "grey60") +
    geom_line() + geom_point(size = 1) +
    facet_wrap(~panel) +
    labs(x = "Steps after the shock j", y = "Response")
}

まず、片方の系列にだけ、大きさ1のショックを入れます。

In [ ]:
plot_irf(irf_theory(Phi))

- $x$ へのショック：$x$ 自身には、0.6倍ずつ小さくなりながら残ります。$y$ には、1時点遅れて伝わり、いったん大きくなってから消えていきます。
- $y$ へのショック：$y$ 自身には残りますが、$x$ にはまったく伝わりません。片方向にだけ効くように作ったからです。

ここで、困ることがあります。ノイズ $w_{x,t}$ と $w_{y,t}$ は、同じ時点どうしで相関しています（相関0.5）。
「$x$ にだけショックが入り、$y$ には入らない」という場面は、このデータでは、ふつう起きません。

そこで、ショックを、互いに無相関な2つに分け直します。これを直交化（orthogonalization）といいます。
分け方は、変数の並べ順で決まります。

- $x$ を先に置く：同じ時点で2本が一緒に動いた分は、$x$ のショックに数える。$x$ のショックは、同じ時点で $y$ も動かす。
- $y$ を先に置く：一緒に動いた分は、$y$ のショックに数える。$y$ のショックは、同じ時点で $x$ も動かす。

In [ ]:
irf_xy <- cbind(irf_theory(Phi, ortho_shock(Sigma, c(1, 2))), order = "x first")
irf_yx <- cbind(irf_theory(Phi, ortho_shock(Sigma, c(2, 1))), order = "y first")
plot_irf(rbind(irf_xy, irf_yx))

- 並べ順を替えると、図が変わります。とくに右上（$y$ へのショック → $x$）が違います。
- $x$ を先に置くと、$y$ へのショックは $x$ に伝わりません。$y$ を先に置くと、伝わるように見えます。
- 同じデータ、同じ係数でも、「同じ時点で一緒に動いた分を、どちらのショックに数えるか」で、答えが変わります。これは、データからは決められません。

推定したモデルのインパルス応答は、`vars::irf()`で出します。変数の並べ順は、データの列の順です。
帯は、ブートストラップ（残差を並べ替えて、系列を作り直す）による95%の区間です。赤い点が、理論の値です。

In [ ]:
set.seed(1)
ir_sim <- vars::irf(fit_sim, n.ahead = 10, ortho = TRUE, ci = 0.95, runs = 200)
plot_irf(irf_long(ir_sim)) +
  geom_point(data = irf_panel(irf_xy), colour = "red", size = 1.5)

### A-6. 予測する

VARモデルは、2本を同時に予測します。$x$ の先の値を、外から与える必要がありません（第5回の回帰との違い）。

第2回の評価方法で、1本だけのモデルと比べます。最後の20時点をテストデータにします。

- `mean`：平均法（ベンチマーク）
- `ar1`：1本だけのAR(1)（第3回）
- `var`：2本のVAR(1)

In [ ]:
h <- 20
train <- sim |> filter(t <= n - h)
sim_one <- as_tsibble(sim_long, key = series, index = t)        # 1本ずつ扱うための、縦長の形
fc_one <- sim_one |> filter(t <= n - h) |>
  model(mean = MEAN(value), ar1 = ARIMA(value ~ 1 + pdq(1, 0, 0))) |>
  forecast(h = h)
fc_var <- train |>
  model(var = VAR(vars(x, y) ~ 1 + AR(1))) |>
  forecast(h = h)
acc_sim <- rbind(accuracy(fc_one, sim_one) |> select(series, .model, RMSE, MAE),
                 accuracy(fc_var, sim) |> select(series = .response, .model, RMSE, MAE))
acc_sim |> arrange(series, .model)

1回の分け方では、たまたまかもしれません。系列を作り直して、500回くり返します。
ここでは、自分で作った関数（`fit_var1`、`forecast_var1`）を使います。4時点先までの予測のRMSEを、平均します。

- `ar1`：自分の過去だけ
- `var2`：$x$ と $y$ の2本
- `var5`：$x$ と $y$ に、無関係なホワイトノイズ3本を加えた5本

訓練データの長さを、40と200の2通りにします。

In [ ]:
set.seed(1)
one_rep <- function(n_train, h = 4) {
  Zr <- sim_var1(matrix(rnorm(2 * (n_train + h + 100)), ncol = 2) %*% chol(Sigma), Phi)
  Z5 <- cbind(Zr, matrix(rnorm(3 * (n_train + h)), ncol = 3))      # 無関係な3本を加える
  tr <- 1:n_train
  te <- n_train + 1:h
  f <- function(Zm) forecast_var1(fit_var1(Zm[tr, , drop = FALSE]), Zm[n_train, ], h)
  c(x_ar1 = rmse(f(Zr[, 1, drop = FALSE]), Zr[te, 1]), x_var2 = rmse(f(Zr)[, 1], Zr[te, 1]),
    x_var5 = rmse(f(Z5)[, 1], Zr[te, 1]),
    y_ar1 = rmse(f(Zr[, 2, drop = FALSE]), Zr[te, 2]), y_var2 = rmse(f(Zr)[, 2], Zr[te, 2]),
    y_var5 = rmse(f(Z5)[, 2], Zr[te, 2]))
}
fc_rep <- rbind(n_train_40  = rowMeans(replicate(500, one_rep(40))),
                n_train_200 = rowMeans(replicate(500, one_rep(200))))
round(fc_rep, 3)

- $y$ の予測：`var2`が`ar1`より良くなります。$x$ の過去が役に立つからです。
- $x$ の予測：`var2`は`ar1`より良くなりません。$y$ の過去は役に立たず、推定する係数が増えただけだからです。
- 無関係な系列を加えた`var5`は、`var2`より悪くなります。訓練データが短いほど、差が大きくなります。

変数を増やせば、必ず良くなるわけではありません。役に立たない変数は、推定の誤差を増やすだけです。

## B. 実データを解析する

`fpp3`パッケージの`us_change`を使います。米国の経済指標の、四半期ごとの変化率です。第5回と同じデータです。

B節は、A節で作った関数（`granger`、`irf_long`、`plot_irf`）を使います。A節を実行してから進んでください。

### B-0. 設定

このセルの値を変えると、B節の解析がすべて変わります。まずはこのまま実行してください。

In [ ]:
# ここを変えると、以下の解析がすべて変わる
var1      <- "Consumption"   # 1本目の系列："Consumption" "Income" "Production" "Savings" "Unemployment" のどれか
var2      <- "Income"        # 2本目の系列：上の5つのうち、var1 と違うもの
year_from <- 1970            # 始まりの年：1970, 1980, 1990 のどれか
ic        <- "BIC"           # 次数を選ぶ規準："AIC" か "BIC"
h         <- 8               # 予測する期の数：4, 8, 12 のどれか

インパルス応答の直交化では、`var1`を先、`var2`を後に置きます。`var1`と`var2`を入れ替えると、並べ順が替わります。

### B-1. データを確かめる

`us_change`は`fpp3`パッケージに入っています。パッケージ全体は読み込まず、データだけを取り出します。

In [ ]:
us_change <- fpp3::us_change
glimpse(us_change)
range(us_change$Quarter)
c(rows = nrow(us_change), missing = sum(is.na(us_change)), gaps = has_gaps(us_change)$.gaps)

- 1行は1四半期です。1970年第1四半期から2019年第2四半期まで、198行あります。欠測も、行の抜けもありません。
- 個人消費（`Consumption`）、可処分所得（`Income`）、生産（`Production`）、貯蓄（`Savings`）の、前の期からの変化率（%）と、失業率の変化（`Unemployment`）です。

### B-2. 前処理

設定で選んだ2本を、`y1`、`y2`という名前にします。最後の`h`期をテストデータにします。

In [ ]:
us <- us_change |>
  filter(format(Quarter, "%Y") >= year_from) |>
  mutate(y1 = .data[[var1]], y2 = .data[[var2]])
unit <- function(v) if (v == "Unemployment") "change" else "% change"
labs2 <- c(y1 = sprintf("%s (%s)", var1, unit(var1)), y2 = sprintf("%s (%s)", var2, unit(var2)))
n <- nrow(us)
train <- us |> filter(Quarter <= max(Quarter) - h)
c(n = n, train = nrow(train), test = n - nrow(train))

In [ ]:
us_long <- data.frame(Quarter = rep(us$Quarter, 2), series = rep(c("y1", "y2"), each = n),
                      value = c(us$y1, us$y2))
ggplot(us_long, aes(Quarter, value)) +
  geom_line() +
  facet_wrap(~series, ncol = 1, scales = "free_y", labeller = as_labeller(labs2)) +
  labs(x = "Quarter", y = NULL)

2本の関係を調べる前に、定常とみなせるかを確かめます（A-2）。第4回のKPSS検定を使います。帰無仮説は「定常である」です。

In [ ]:
kpss <- rbind(cbind(series = var1, us |> features(y1, unitroot_kpss)),
              cbind(series = var2, us |> features(y2, unitroot_kpss)))
kpss

この系列は、もとの量（消費額など）の変化率です。変化率は、もとの量の差分にあたります。
「差分をとって定常に直す」処理が、すでに済んでいます。p値が0.05より大きければ、定常とみなして進みます。

（`kpss_pvalue`は、0.01より小さいときは0.01、0.1より大きいときは0.1と表示されます。）

### B-3. 相互相関

In [ ]:
us |>
  CCF(y1, y2, lag_max = 8) |>
  autoplot() +
  labs(x = "Lag h (quarter)", y = sprintf("Sample CCF: Corr(%s_t, %s_{t+h})", var1, var2))

正のラグは`var1`が先、負のラグは`var2`が先です。どちらの側にも、破線の外に出るラグがあるかを見ます。
ただし、相互相関には、それぞれの自己相関が混ざっています（A-2）。どちらが先かは、次のVARモデルで調べます。

### B-4. VARモデルを当てはめる

次数を、設定で選んだ規準（`ic`）で選びます。候補は1から4までです。

In [ ]:
d_train <- as.data.frame(train)[, c(var1, var2)]
sel <- vars::VARselect(d_train, lag.max = 4, type = "const")
round(sel$criteria[c("AIC(n)", "SC(n)"), ], 3)
p <- unname(sel$selection[if (ic == "AIC") "AIC(n)" else "SC(n)"])
p

In [ ]:
fit <- vars::VAR(d_train, p = p, type = "const")
round(vars::Bcoef(fit), 3)
root_max <- max(vars::roots(fit))        # 固有値の絶対値の最大。1より小さければ、定常
res_cor <- cor(resid(fit))[1, 2]         # 2本の残差の、同じ時点どうしの相関
c(root_max = root_max, res_cor = res_cor)

- 行が式（どの系列を説明するか）、列が係数です。`Income.l1`は、所得の1期前の値の係数です。
- `root_max`が1より小さければ、推定したモデルは定常です。
- `res_cor`は、ノイズの同じ時点どうしの相関です。0から遠いほど、インパルス応答が並べ順に左右されます（A-5）。

残差に自己相関が残っていないかを、第4回のリュング・ボックス検定で確かめます（式ごと、ラグ8まで）。

In [ ]:
res <- tsibble(Quarter = rep(train$Quarter[-(1:p)], 2),
               series = rep(c(var1, var2), each = nrow(resid(fit))),
               resid = as.numeric(resid(fit)), key = series, index = Quarter)
res |> ACF(resid, lag_max = 12) |> autoplot() +
  labs(x = "Lag h (quarter)", y = "Sample ACF of residuals")
lb <- res |> features(resid, ljung_box, lag = 8)
lb

### B-5. Granger因果

In [ ]:
gr <- rbind(granger(fit, var1), granger(fit, var2))
gr$effect <- c(var2, var1)                 # cause の過去が、effect の予測に役立つか
gr[, c("cause", "effect", "F", "df1", "df2", "p_value")]

p値が0.05より小さければ、「`cause`の過去は、`effect`の予測に役立つ」と読みます。原因である、とは読みません。

### B-6. インパルス応答

`var1`を先に置いた場合と、`var2`を先に置いた場合を、重ねて描きます。帯は95%の区間です。

In [ ]:
set.seed(1)
ir1 <- vars::irf(fit, n.ahead = 8, ortho = TRUE, ci = 0.95, runs = 200)
fit_rev <- vars::VAR(d_train[, c(var2, var1)], p = p, type = "const")   # 並べ順を逆にする
ir2 <- vars::irf(fit_rev, n.ahead = 8, ortho = TRUE, ci = 0.95, runs = 200)
irf_us <- rbind(cbind(irf_long(ir1), order = sprintf("%s first", var1)),
                cbind(irf_long(ir2), order = sprintf("%s first", var2)))
plot_irf(irf_us) +
  labs(y = "Response to a one-s.d. shock") +
  theme(legend.position = "bottom")

- ショックの大きさは、標準偏差1つ分です。
- 2つの色が大きく離れているパネルは、並べ順で答えが変わるところです。

### B-7. 予測を比べる

最後の`h`期を、5つのモデルで予測します。

- `mean`、`naive`：第2回のベンチマーク（四半期の変化率で、季節の上下が見えないので、季節ナイーブ法は使いません）
- `arima`：1本だけのARMAモデル（第3回。次数は、第4回・第5回と同じく、`ARIMA()`がAICcの小さいものを選ぶ。変化率なので、差分はとらない）
- `var2`：選んだ2本のVAR($p$)
- `var5`：`us_change`の5本すべてを使ったVAR($p$)

In [ ]:
us_one <- as_tsibble(us_long, key = series, index = Quarter)       # 1本ずつ扱うための、縦長の形
fit_one <- us_one |> filter(Quarter <= max(Quarter) - h) |>
  model(mean  = MEAN(value),
        naive = NAIVE(value),
        arima = ARIMA(value ~ pdq(d = 0) + PDQ(0, 0, 0)))
fit_v2 <- train |> model(var2 = VAR(vars(y1, y2) ~ 1 + AR(p)))
fit_v5 <- train |>
  model(var5 = VAR(vars(Consumption, Income, Production, Savings, Unemployment) ~ 1 + AR(p)))
# モデルごとの誤差を、同じ形の表にまとめる（y1 と y2 の行だけを残す）
acc_table <- function(a_one, a_v2, a_v5) {
  a_v5$.response <- c("y1", "y2")[match(a_v5$.response, c(var1, var2))]
  rbind(a_one |> select(series, .model, RMSE, MAE),
        a_v2 |> select(series = .response, .model, RMSE, MAE),
        a_v5 |> filter(!is.na(.response)) |> select(series = .response, .model, RMSE, MAE)) |>
    mutate(series = ifelse(series == "y1", var1, var2)) |>
    arrange(series, .model)
}
acc <- acc_table(accuracy(forecast(fit_one, h = h), us_one),
                 accuracy(forecast(fit_v2, h = h), us),
                 accuracy(forecast(fit_v5, h = h), us))
acc

`var1`の予測を、図にします。

In [ ]:
fc_v2 <- forecast(fit_v2, h = h)
fc_line <- rbind(
  forecast(fit_one, h = h) |> as_tibble() |> filter(series == "y1", .model != "naive") |>
    select(Quarter, .model, value = .mean),
  data.frame(Quarter = fc_v2$Quarter, .model = "var2", value = fc_v2$.mean[, "y1"]))
ggplot(us |> filter(Quarter > max(Quarter) - 5 * h), aes(Quarter, y1)) +
  geom_line() +
  geom_line(data = fc_line, aes(Quarter, value, colour = .model), linewidth = 0.8) +
  labs(x = "Quarter", y = labs2[["y1"]], colour = "Model")

### B-8. 時系列交差検証

訓練データを`h`期ずつ伸ばしながら、同じ評価を6回くり返します。次数 $p$ は、B-4で選んだものを使います。

In [ ]:
cv_one <- us_one |> filter(Quarter <= max(Quarter) - h) |>
  stretch_tsibble(.init = n - 6 * h, .step = h)
cv_wide <- train |> stretch_tsibble(.init = n - 6 * h, .step = h)
acc_cv <- acc_table(
  cv_one |>
    model(mean  = MEAN(value),
          naive = NAIVE(value),
          arima = ARIMA(value ~ pdq(d = 0) + PDQ(0, 0, 0))) |>
    forecast(h = h) |> accuracy(us_one),
  cv_wide |> model(var2 = VAR(vars(y1, y2) ~ 1 + AR(p))) |>
    forecast(h = h) |> accuracy(us),
  cv_wide |>
    model(var5 = VAR(vars(Consumption, Income, Production, Savings, Unemployment) ~ 1 + AR(p))) |>
    forecast(h = h) |> accuracy(us))
acc_cv

読み方：

- B-3、B-5：相互相関の山の位置と、Granger因果の検定の結果は、同じ向きを指しているか。
- B-5：Granger因果が「ある」と出ても、原因だとは言えません。両方を動かす別の系列（景気など）があるかもしれません（A-4）。
- B-6：並べ順で図が変わるパネルは、「同じ期のうちに一緒に動いた分」をどちらのショックに数えるかで、答えが変わるところです。
- B-7、B-8：Granger因果が「ある」と出た系列でも、`var2`が`arima`や`mean`に勝つとは限りません。`var5`が`var2`より良いとも限りません（A-6）。

## C. 設定を変えて回し直す

B-0の設定セルを書き替えて、B節を上からすべて実行し直してください
（Colabでは、設定セルを選んで、メニューの［ランタイム］→［以降のセルを実行］）。

変えられる値：

| 変数 | 値 |
| --- | --- |
| `var1` | `"Consumption"` `"Income"` `"Production"` `"Savings"` `"Unemployment"` |
| `var2` | 上の5つのうち、`var1`と違うもの |
| `year_from` | 1970、1980、1990 |
| `ic` | `"AIC"` `"BIC"` |
| `h` | 4、8、12 |

読むときの観点：

1. `var1`と`var2`を入れ替えると、インパルス応答（B-6）の2つの色は、どう入れ替わるか。Granger因果の検定（B-5）は変わるか。
2. 規準を`"AIC"`にすると、次数 $p$ はいくつになるか。Granger因果のp値と、予測の誤差は、どちらへ動くか。
3. 系列の組を替えると、Granger因果が両方向に出る組、片方向だけの組、どちらにも出ない組は、それぞれどれか。
4. `var2`が`arima`に勝つのは、どの組か。Granger因果が出た向きと、合っているか。`var5`が`var2`に勝つ組はあるか。

正解は1つではありません。自分の設定で出た結果を、最初の設定（Consumption と Income、1970年から、BIC、8期先）の結果と比べてください。